<a href="https://colab.research.google.com/github/danieligelnik/CCFraudProject/blob/main/Test_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install dependencies as needed:
%pip install kagglehub[pandas-datasets]

In [2]:
!pip install import-ipynb
import import_ipynb
import requests
import os

# 1. URL to the RAW version of the notebook on GitHub
github_url = "https://raw.githubusercontent.com/danieligelnik/CCFraudProject/main/help_functions.ipynb"
# Corrected the typo in the filename here:
notebook_filename = "help_functions.ipynb"

# 2. Download the notebook file locally
response = requests.get(github_url)
with open(notebook_filename, 'wb') as f:
    f.write(response.content)

# 3. Import the notebook as a module
try:
    # The module name must match the filename (without .ipynb)
    from help_functions import *
    #from help_functions import load_kagglehub_dataset
    print(f"Successfully imported functions from {notebook_filename}")
except Exception as e:
    print(f"Error importing notebook: {e}")

Successfully imported functions from help_functions.ipynb


In [3]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import holidays as hol
from sklearn.model_selection import train_test_split, KFold

# **Data**

### Download data set & removed is_fraud category

In [5]:
# 1. Download the latest version of the dataset
dataset_path = kagglehub.dataset_download("dermisfit/fraud-transactions-dataset")

# 2. Construct the full local path to the specific CSV file
credit_cards_path_test = "fraudTest.csv"
full_file_path = f"{dataset_path}/{credit_cards_path_test}"

# 3. Load the dataset using pandas from the local path
df_cards_test = pd.read_csv(full_file_path)

# 4. Remove the 'is_fraud' column as requested
df_cards_test = df_cards_test.drop(columns=['is_fraud'])

# Verify the column is removed and show data info
# Calling df_info directly since it was imported via 'from help_functions import *'
df_info(df_cards_test, 'head')

Using Colab cache for faster access to the 'fraud-transactions-dataset' dataset.
First 5 records:


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long
0,0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,Columbia,SC,29209,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714
1,1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,Altonah,UT,84002,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431
2,2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,Bellmore,NY,11710,40.6729,-73.5365,34496,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111
3,3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,Titusville,FL,32780,28.5697,-80.8191,54767,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1371816915,28.812398,-80.883061
4,4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,Falmouth,MI,49632,44.2529,-85.0170,1126,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1371816917,44.959148,-85.884734


**Data encoding**

**Feature engineering**

In [7]:
def feature_engineering(df):
    df_new = df.copy()
    df_new['gender'] = df['gender'].map({'F': 1, 'M': 0})
    df_new = pd.get_dummies(df_new, columns=['category'], prefix='category', drop_first=True, dtype=int)
    df_new['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
    df_new['age'] = (df_new['trans_date_trans_time'] - pd.to_datetime(df.dob)).dt.days
    df_new['is_weekend'] = df_new['trans_date_trans_time'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)
    df_new['trans_date'] = df_new['trans_date_trans_time'].dt.date.apply(lambda x: x.toordinal()).astype(np.uint64)

    # If extract_features is defined in the imported namespace, this will work:
    try:
        df_new = extract_features(df_new)
    except NameError:
        print("Warning: extract_features not found, proceeding with basic features.")

    return df_new

# Run the engineering process
df_test_eng = feature_engineering(df_cards_test)
print("Feature engineering complete!")
df_info(df_test_eng, 'head')

Feature engineering complete!
First 5 records:


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,category_food_dining,category_gas_transport,category_grocery_net,category_grocery_pos,...,category_shopping_pos,category_travel,age,is_weekend,trans_date,merch_feat_0,merch_feat_1,merch_feat_2,merch_feat_3,merch_feat_4,merch_feat_5,merch_feat_6,merch_feat_7,merch_feat_8,merch_feat_9,job_feat_0,job_feat_1,job_feat_2,job_feat_3,job_feat_4,job_feat_5,job_feat_6,job_feat_7,job_feat_8,job_feat_9
0,0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,2.86,Jeff,Elliott,0,351 Darlene Green,Columbia,SC,29209,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0,0,0,0,...,0,0,19087,1,737597,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,29.84,Joanne,Williams,1,3638 Marsh Union,Altonah,UT,84002,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0,0,0,0,...,0,0,11113,1,737597,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",41.28,Ashley,Lopez,1,9333 Valentine Point,Bellmore,NY,11710,40.6729,-73.5365,34496,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0,0,0,0,...,0,0,18141,1,737597,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0
3,3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,60.05,Brian,Williams,0,32941 Krystal Mill Apt. 552,Titusville,FL,32780,28.5697,-80.8191,54767,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1371816915,28.812398,-80.883061,0,0,0,0,...,0,0,12020,1,737597,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,3.19,Nathan,Massey,0,5783 Evan Roads Apt. 465,Falmouth,MI,49632,44.2529,-85.0170,1126,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1371816917,44.959148,-85.884734,0,0,0,0,...,0,1,23727,1,737597,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


**Dropping redundant features**

In [8]:
df_test_eng = df_test_eng.drop(columns=['trans_date_trans_time','Unnamed: 0','merchant','long','job','trans_num','unix_time','merch_long', 'dob', 'city', 'state','first','last','street'])